# Tesla Valuation Model Based on AI Storytelling Factor

This notebook walks through the complete Tesla stock valuation model pipeline:

1. **Data collection** – price history and quarterly fundamentals (yfinance, with synthetic fallback)
2. **NLP narrative factors** – news sentiment, earnings-call tone, social media sentiment
3. **Feature engineering** – fundamental, technical, and company-level factors
4. **LASSO variable selection** – screen for core predictive variables
5. **Comparative model fitting** – Random Forest, Bagging, Boosting (XGBoost), Neural Network
6. **Evaluation** – CV and test-set comparison to select the optimal model


In [ ]:
import sys
from pathlib import Path

# Add the project root to PYTHONPATH
sys.path.insert(0, str(Path.cwd().parent))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='tab10')
%matplotlib inline

## 1. Data Collection

In [ ]:
from src.data_collection import fetch_price_data, fetch_financial_data

price_df = fetch_price_data(start='2018-01-01', end='2024-12-31')
fundamental_df = fetch_financial_data()

print(f'Price data: {len(price_df):,} trading days')
print(f'Fundamental data: {len(fundamental_df)} quarters')
price_df.tail(3)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(price_df.index, price_df['Close'], linewidth=1, color='steelblue')
axes[0].set_ylabel('Close Price (USD)')
axes[0].set_title('Tesla (TSLA) – Close Price 2018–2024')

axes[1].bar(price_df.index, price_df['Volume'] / 1e6, width=1, color='gray', alpha=0.6)
axes[1].set_ylabel('Volume (millions)')
axes[1].set_xlabel('Date')

plt.tight_layout()
plt.show()

## 2. NLP Narrative Factors

In [ ]:
from src.nlp_factors import build_nlp_factors

nlp_df = build_nlp_factors(price_df)
nlp_df[['NLP_news_sentiment', 'NLP_earnings_sentiment', 'NLP_social_sentiment',
         'NLP_combined_sentiment']].describe().round(3)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

channels = [
    ('NLP_news_sentiment',     'News Sentiment',          'royalblue'),
    ('NLP_earnings_sentiment', 'Earnings-Call Sentiment', 'darkorange'),
    ('NLP_social_sentiment',   'Social Sentiment',        'seagreen'),
]

for ax, (col, label, color) in zip(axes, channels):
    ax.plot(nlp_df.index, nlp_df[col], linewidth=0.8, color=color, label=label)
    ax.axhline(0, color='black', linewidth=0.7, linestyle='--')
    ax.fill_between(nlp_df.index, nlp_df[col], 0,
                    where=nlp_df[col] >= 0, alpha=0.2, color='green')
    ax.fill_between(nlp_df.index, nlp_df[col], 0,
                    where=nlp_df[col] < 0, alpha=0.2, color='red')
    ax.set_ylabel('Score [-1,1]')
    ax.legend(loc='upper left', fontsize=9)

axes[-1].set_xlabel('Date')
fig.suptitle('Tesla – NLP / AI Narrative Sentiment Factors', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
from src.feature_engineering import build_feature_matrix

feature_df = build_feature_matrix(
    price_df=price_df,
    fundamental_df=fundamental_df,
    nlp_df=nlp_df,
    target_horizon=21,
)
print(f'Feature matrix: {feature_df.shape[0]:,} rows x {feature_df.shape[1]} columns')

factor_groups = {
    'Technical':     [c for c in feature_df.columns if any(x in c for x in ['MA_', 'RSI', 'MACD', 'BB_', 'HV_', 'ATR', 'Momentum', 'Volume'])],
    'Fundamental':   [c for c in feature_df.columns if any(x in c for x in ['PE_', 'PS_', 'PB_', 'EV', 'Margin', 'ROE', 'ROA', 'Revenue', 'Debt'])],
    'Company-level': [c for c in feature_df.columns if any(x in c for x in ['Beta', 'Corr', 'Market_Cap', 'Log_Market', 'Short', 'Price_jump'])],
    'NLP':           [c for c in feature_df.columns if c.startswith('NLP_')],
}
for g, cols in factor_groups.items():
    print(f'{g}: {len(cols)} features')

In [ ]:
# Correlation heat-map (sample 30 features)
sample_cols = list(feature_df.select_dtypes(include=np.number).columns[:30])
corr = feature_df[sample_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(corr, ax=ax, cmap='RdYlGn', center=0, annot=False,
            linewidths=0.3, cbar_kws={'label': 'Pearson r'})
ax.set_title('Factor Correlation Heat Map (first 30 features)', fontsize=12, fontweight='bold')
ax.tick_params(labelsize=6)
plt.tight_layout()
plt.show()

## 4. Train / Test Split

In [ ]:
df = feature_df.dropna(thresh=int(feature_df.shape[1] * 0.5))
X = df.drop(columns=['Target'])
y = df['Target']

split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f'Training: {len(X_train):,} samples  ({X_train.index[0].date()} to {X_train.index[-1].date()})')
print(f'Test:     {len(X_test):,} samples  ({X_test.index[0].date()} to {X_test.index[-1].date()})')
print(f'Features: {X_train.shape[1]}')

## 5. LASSO Variable Selection + Baseline Model

In [ ]:
from src.models import fit_lasso

lasso_result = fit_lasso(X_train, y_train, X_test, y_test)
print(f'LASSO selected {len(lasso_result.selected_features)} features:')
print(lasso_result.selected_features)
print(f'\nCV R²  = {lasso_result.cv_r2:.4f}')
print(f'Test R² = {lasso_result.test_r2:.4f}')

In [ ]:
# Top-20 features by LASSO coefficient magnitude
top20 = lasso_result.feature_importance.head(20)
fig, ax = plt.subplots(figsize=(9, 6))
top20.sort_values().plot.barh(ax=ax, color='steelblue', alpha=0.85)
ax.set_title('Top 20 Features – LASSO Coefficient Magnitude', fontweight='bold')
ax.set_xlabel('|Coefficient|')
plt.tight_layout()
plt.show()

In [ ]:
# LASSO regularisation path
from sklearn.linear_model import lasso_path
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

imp = SimpleImputer(strategy='median')
scl = StandardScaler()
Xp = scl.fit_transform(imp.fit_transform(X_train.values))
alphas, coefs, _ = lasso_path(Xp, y_train.values, n_alphas=80, max_iter=5000)

fig, ax = plt.subplots(figsize=(12, 5))
for coef_path in coefs:
    ax.plot(-np.log10(alphas + 1e-12), coef_path, linewidth=0.8, alpha=0.7)
ax.set_xlabel('-log10(alpha)  ->  decreasing regularisation')
ax.set_ylabel('Coefficient value')
ax.set_title('LASSO Regularisation Path – Tesla Valuation Features', fontweight='bold')
ax.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

## 6. Comparative Model Fitting

In [ ]:
from src.models import fit_all_models
from src.evaluation import summary_table, print_summary

results = fit_all_models(X_train, y_train, X_test, y_test)
print_summary(results)

## 7. Evaluation and Model Selection

In [ ]:
tbl = summary_table(results).reset_index()
models = tbl['Model'].tolist()
x = np.arange(len(models))
width = 0.3

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Tesla Valuation Model – Performance Comparison', fontsize=13, fontweight='bold')

ax = axes[0]
ax.bar(x - width/2, tbl['CV R2'], width, label='CV R2', alpha=0.85)
ax.bar(x + width/2, tbl['Test R2'], width, label='Test R2', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(models, rotation=25, ha='right', fontsize=8)
ax.set_ylabel('R2'); ax.set_title('R2 Score (higher = better)')
ax.legend(); ax.axhline(0, color='black', linewidth=0.8, linestyle='--')

ax = axes[1]
ax.bar(x - width/2, tbl['CV RMSE'], width, label='CV RMSE', alpha=0.85)
ax.bar(x + width/2, tbl['Test RMSE'], width, label='Test RMSE', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(models, rotation=25, ha='right', fontsize=8)
ax.set_ylabel('RMSE'); ax.set_title('RMSE (lower = better)')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Predicted vs actual for each model
ncols = 3
nrows = (len(results) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(15, nrows*5))
axes_flat = axes.flatten()
palette = sns.color_palette('tab10')

for i, r in enumerate(results):
    ax = axes_flat[i]
    y_pred = r.pipeline.predict(X_test[r.selected_features].values)
    ax.scatter(y_test.values, y_pred, alpha=0.3, s=10, color=palette[i])
    lim = max(abs(y_test).max(), abs(y_pred).max()) * 1.05
    ax.plot([-lim, lim], [-lim, lim], 'r--', linewidth=1.2)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')
    ax.set_title(f'{r.name}\nTest R2={r.test_r2:.4f}', fontsize=9)

for j in range(len(results), len(axes_flat)):
    axes_flat[j].set_visible(False)

fig.suptitle('Predicted vs Actual 21-day Forward Log-Returns', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
tbl_final = summary_table(results)
best_model = tbl_final['Test R²'].idxmax()
print(f'Optimal model: {best_model}')
print(f'Test R2   = {tbl_final.loc[best_model, "Test R²"]:.4f}')
print(f'Test RMSE = {tbl_final.loc[best_model, "Test RMSE"]:.6f}')